# [센서 실습 14] 수위 센서 – JSN-SR04T 방수 초음파 센서

물탱크 뚜껑에 단 **방수 초음파 센서 JSN-SR04T**로 수면까지의 거리를 재서 수위를 계산하고, 만수위·저수위 경보와 펌프 보호 로직을 만듭니다.

**이 노트북의 흐름**
1. 센서의 원리와 부품 사양
2. 라즈베리파이 연결 방법
3. 실제 센서를 읽는 코드 (라즈베리파이용)
4. 장비가 없을 때 쓰는 **가상 센서** (같은 코드로 실행)
5. 수집 → 판정 → 시각화 → 저장·IoT 전송

> 장비가 없어도 `USE_REAL_SENSOR = False` 상태 그대로 위에서부터 `Shift + Enter`로 모두 실행됩니다.

## 1. 센서의 원리
수위 센서는 물에 닿는 방식(전극식, 플로트식)과 **닿지 않는 방식(초음파·레이더)** 이 있습니다.
JSN-SR04T는 HC-SR04와 원리가 같은 **방수형 초음파 센서**로, 탱크 위에서 수면까지 거리를 잽니다.

$$\text{수위} = \text{센서~바닥 거리(탱크 높이)} - \text{센서~수면 거리}$$

- 물에 닿지 않아 오염·부식 걱정이 없음
- 센서 바로 앞 약 **20cm는 측정 불가(사각지대)** → 만수위가 센서에서 25cm 이상 떨어지게 설치
- 탱크 벽에 반사된 소리가 섞이지 않도록 센서는 탱크 중앙에 설치

### 부품 사양
| 항목 | 내용 |
|---|---|
| 측정 거리 | 25 ~ 450 cm (사각지대 약 20cm) |
| 정확도 | 약 ±1 cm |
| 방수 | 센서 헤드 방수 |
| 전원 | 5V |
| 출력 | TRIG/ECHO 펄스 (HC-SR04 호환) |

## 2. 라즈베리파이 연결 방법
**준비물**: JSN-SR04T(센서+제어보드), 저항 1kΩ·2kΩ, 점퍼선

| 연결할 곳 | 라즈베리파이 | 설명 |
|---|---|---|
| JSN-SR04T 5V / GND | 5V (2번 핀) / GND |  |
| JSN-SR04T TRIG | GPIO23 (16번 핀) |  |
| JSN-SR04T ECHO → 1kΩ → GPIO24 | GPIO24 (18번 핀) | GPIO24 ↔ GND 사이 2kΩ (5V→3.3V) |

> ⚠️ ECHO 출력이 5V이므로 HC-SR04와 마찬가지로 **전압 분배**가 필요합니다.

**인터페이스 활성화**
- **GPIO**: 별도 설정 없이 바로 사용 가능

**라이브러리 설치 (라즈베리파이 터미널)**
```bash
pip3 install gpiozero
```

## 3. 공통 준비 – 한글 폰트와 라이브러리

In [ ]:
# Colab에서 그래프의 한글이 깨지지 않도록 나눔고딕 폰트를 설정합니다. (처음 한 번 실행)
import os, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(font_path):
    subprocess.run("apt-get -qq -y install fonts-nanum", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
import time, json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 4. 실제 센서 읽기 (라즈베리파이용)
### 4-1. 원시값 → 물리량 변환식 (실제·가상 공통)
에코 시간 → 거리 → 수위로 바꿉니다. 사각지대(22cm 미만)에서 나온 값은 믿을 수 없으므로 `None`으로 처리합니다.

In [ ]:
TANK_HEIGHT_CM = 150      # 센서에서 탱크 바닥까지 거리
BLIND_CM = 22             # 사각지대

def echo_to_distance_cm(pulse_s, temp_c=20.0):
    if pulse_s is None:
        return None
    return pulse_s * (331.3 + 0.606 * temp_c) * 100 / 2

def distance_to_level(distance_cm):
    if distance_cm is None or distance_cm < BLIND_CM:
        return None
    return round(max(0.0, TANK_HEIGHT_CM - distance_cm), 1)

print(distance_to_level(echo_to_distance_cm(0.00583)), "cm")    # 거리 100cm → 수위 50cm

### 4-2. 실제 센서 클래스
HC-SR04와 같은 방법으로 에코 시간을 재고, 수위로 바꿉니다. 물결 때문에 값이 흔들리므로 5회 측정해 중앙값을 씁니다.

> 이 셀은 **클래스를 정의만** 합니다. 하드웨어 라이브러리는 객체를 만들 때(`__init__`) 불러오므로 Colab에서 실행해도 오류가 나지 않습니다. 실제 동작은 라즈베리파이에서 `USE_REAL_SENSOR = True`로 해야 합니다.

In [ ]:
class RealJSNSR04T:
    def __init__(self, trig=23, echo=24):
        from gpiozero import DigitalOutputDevice, DigitalInputDevice
        self.trig, self.echo = DigitalOutputDevice(trig), DigitalInputDevice(echo)

    def _pulse(self, timeout=0.04):
        self.trig.on(); time.sleep(0.00002); self.trig.off()     # JSN-SR04T는 20µs 권장
        t0 = time.perf_counter()
        while not self.echo.value:
            if time.perf_counter() - t0 > timeout: return None
        start = time.perf_counter()
        while self.echo.value:
            if time.perf_counter() - start > timeout: return None
        return time.perf_counter() - start

    def read(self):
        pulses = [p for p in (self._pulse() for _ in range(5)) if p] or [None]
        d = echo_to_distance_cm(float(np.median(pulses)) if pulses[0] else None)
        return {"distance_cm": d, "level_cm": distance_to_level(d)}

## 5. 장비가 없을 때 – 가상 센서
### 왜 가상 센서를 쓰나요?
실제 센서의 값은 다음 순서로 만들어집니다.

```
① 전기 신호 (전압, 펄스, 0/1)  →  ② 원시값 raw (ADC 숫자, 바이트, 문자열)  →  ③ 변환식  →  ④ 물리량 (°C, ppm, cm …)
```
가상 센서는 ①을 건너뛰고 **②원시값을 실제 센서와 똑같은 형식으로** 만들어 줍니다.
따라서 ③변환식과 그 뒤의 판정·시각화·저장 코드는 실제 장비와 가상 장비에서 **완전히 같은 코드**를 사용합니다.

```
[장비 있음]  센서 ──배선──▶ 라즈베리파이 ──▶ Real….read()  ─┐
                                                             ├──▶ 변환식 → 판정 → 그래프 → 저장/전송
[장비 없음]  가정한 상황(시나리오) ──▶ Virtual….read() ──────┘
```
나중에 장비를 연결하면 `USE_REAL_SENSOR = True` **한 줄만** 바꾸면 됩니다.

### 가상 값은 이렇게 정했습니다
| 항목 | 가정한 값 | 근거 |
|---|---|---|
| 탱크 | 높이 150cm 빗물 저장 탱크 | 옥상·텃밭 빗물 탱크 |
| 처음 수위 | 40cm | |
| 비 | 2~7시 시간당 최대 30cm 유입 | 집중 호우 |
| 사용 | 9시부터 시간당 약 10cm 사용 (텃밭 물주기) | |
| 물결 | 수면 ±1.5cm 흔들림 | 유입 시 물결 |
| 사각지대 | 수면이 센서 22cm 안쪽이면 엉뚱한 짧은 거리 반환 | 실제 센서 특성 |

가상 센서는 수위로부터 **에코 펄스 시간**을 만들어 실제와 같은 변환식에 넣습니다.

In [ ]:
class VirtualJSNSR04T:
    def __init__(self, dt=300, seed=14):
        self.dt, self.t, self.level = dt, 0.0, 40.0
        self.rng = np.random.default_rng(seed)

    def read(self):
        h = self.t / 3600
        inflow = 30 * np.exp(-((h - 4.5) / 1.5) ** 2) if 2 <= h < 7 else 0     # 비 (cm/h)
        outflow = 10 if h >= 9 and self.level > 15 else 0                       # 사용 (cm/h)
        self.level = min(TANK_HEIGHT_CM, max(0.0, self.level + (inflow - outflow) * self.dt / 3600))
        d = TANK_HEIGHT_CM - self.level + self.rng.normal(0, 1.5)
        if d < BLIND_CM:
            d = self.rng.uniform(5, 20)                     # 사각지대에서는 엉뚱한 값
        pulse = 2 * (d / 100) / 343.4
        self.t += self.dt
        dist = echo_to_distance_cm(pulse)
        return {"distance_cm": round(dist, 1), "level_cm": distance_to_level(dist)}

In [ ]:
# 가상 센서 동작 확인 – read()를 3번 호출
test = VirtualJSNSR04T(dt=300)
for _ in range(3):
    print(test.read())

## 6. 센서 선택과 데이터 수집
5분 간격으로 18시간 측정합니다.

- 실제 장비는 `time.sleep(INTERVAL)`으로 측정 간격만큼 기다립니다.
- 가상 센서는 기다리지 않고 시간을 건너뛰므로, 몇 시간·며칠 분량의 데이터도 바로 만들어집니다.

In [ ]:
USE_REAL_SENSOR = False      # ← 라즈베리파이에 센서를 연결했다면 True 로 변경
N_SAMPLES = 216               # 측정 횟수
INTERVAL = 300               # 측정 간격(초)

sensor = RealJSNSR04T() if USE_REAL_SENSOR else VirtualJSNSR04T(dt=INTERVAL)

records = []
for i in range(N_SAMPLES):
    value = sensor.read()
    records.append({"time_s": round(i * INTERVAL, 4), **value})
    if USE_REAL_SENSOR:
        time.sleep(INTERVAL)

df = pd.DataFrame(records)
print(f"{len(df)}개 측정 완료 (총 {N_SAMPLES * INTERVAL / 60:.1f}분 분량)")
df.head()

In [ ]:
df.describe().round(3)

## 7. 판정 – 이벤트 감지
| 수위 | 판정 | 조치 |
|---|---|---|
| 115cm 이상 | 만수위 경고 | 유입 밸브 잠금, 넘침 주의 |
| 20cm 미만 | 저수위 | 펌프 정지 (헛돌면 고장) |
| 측정 불가(`None`) | 사각지대 | 센서 설치 위치 점검 |

In [ ]:
df["hour"] = df["time_s"] / 3600
HIGH, LOW = 115, 20
print("사각지대(측정 불가):", df["level_cm"].isna().sum(), "회")
lv = df["level_cm"].interpolate().bfill()
df["state"] = np.where(lv >= HIGH, "만수위", np.where(lv < LOW, "저수위", "정상"))
for st in ["만수위", "저수위"]:
    part = df[df["state"] == st]
    if len(part):
        print(f"{st}: {part['hour'].min():.1f} ~ {part['hour'].max():.1f}시")
print(f"최고 수위 {lv.max():.0f}cm / 탱크 높이 {TANK_HEIGHT_CM}cm → 저수량 약 {lv.max() / TANK_HEIGHT_CM * 100:.0f}%")

## 8. 시각화

In [ ]:
plt.figure(figsize=(10, 4.5))
plt.plot(df["hour"], df["level_cm"], "k-", lw=1.5, label="수위")
plt.axhline(HIGH, color="gray", ls="--", label=f"만수위 {HIGH}cm")
plt.axhline(LOW, color="gray", ls=":", label=f"저수위 {LOW}cm")
plt.axhspan(TANK_HEIGHT_CM - BLIND_CM, TANK_HEIGHT_CM, color="lightgray", label="센서 사각지대")
plt.ylim(0, TANK_HEIGHT_CM); plt.title("JSN-SR04T 물탱크 수위")
plt.xlabel("시각 (시)"); plt.ylabel("수위 (cm)"); plt.grid(alpha=0.3); plt.legend(loc="upper right"); plt.show()

## 9. 저장과 IoT 전송
측정값은 CSV 파일로 저장하고, IoT 서버로 보낼 때는 보통 **JSON** 형식의 메시지를 **MQTT** 프로토콜로 전송합니다.
(Colab에서는 왼쪽 📁 메뉴에서 저장된 CSV를 내려받을 수 있습니다.)

In [ ]:
csv_name = "액세수위센서_data.csv"
df.to_csv(csv_name, index=False, encoding="utf-8-sig")
print("CSV 저장:", csv_name)

def to_jsonable(v):
    if isinstance(v, (bool, np.bool_)): return bool(v)
    if isinstance(v, (int, np.integer)): return int(v)
    if isinstance(v, (float, np.floating)): return None if np.isnan(v) else round(float(v), 3)
    return str(v)

last = df.iloc[-1].to_dict()
payload = json.dumps({"device": "rpi-01", "sensor": "JSN-SR04T 방수 초음파 센서",
                      "time": datetime.now().isoformat(timespec="seconds"),
                      "values": {k: to_jsonable(v) for k, v in last.items()}}, ensure_ascii=False)
print("IoT 서버로 보낼 메시지(JSON):")
print(payload)

# ▼ 실제로 MQTT 브로커에 보내려면 (pip install paho-mqtt) 주석을 해제하세요.
# import paho.mqtt.client as mqtt
# client = mqtt.Client()
# client.connect("broker.hivemq.com", 1883)          # 공개 테스트 브로커
# client.publish("myclass/rpi-01/water_level", payload)
# client.disconnect()

## 직접 해 보기
- 비의 양(30)을 40으로 늘리면 사각지대에 들어가는지 확인해 보세요. 어떻게 설치를 바꾸면 될까요?
- 탱크 지름이 80cm라면 수위(cm)를 물의 양(리터)으로 바꿔 보세요. (π × 40² × 수위 ÷ 1000)
- 만수위가 되면 유입 밸브를 잠그는 제어를 반복문 안에 넣어 보세요.